# Dataset

In the very beginning you have to obtain a dataset to train your diffusion policy. For this you have several options:

1. Choose the standard Square_D0 / _D1 or _D2 datasets provided by mimicgen. (Download from mimicgen or use the provided)
2. Create your own dataset using augmentation.

Make sure you have your conda enviroment activated for the following commands to run.
(When using VSCode, select the respective conda env. as a kernel)

## 1 Download vanilla dataset

Mimicgen already provides a set of basic datasets. They can be explored on their website: https://mimicgen.github.io/

They can be downloaded here: https://mimicgen.github.io/docs/datasets/mimicgen_corl_2023.html using mimicgen's `download_datasets.py`.
Eg.:

In [2]:

# download core datasets for square D0, D1, D2
!python mimicgen/mimicgen/scripts/download_datasets.py --dataset_type core --tasks square_d0 square_d1 square_d2

The obtained dataset can now be used to start training your diffusion policy to solve the choosen task. See section 3 of this notebook to continue.
</br></br>Alternatively, if you want to augment or change the downloaded datasets, see the next section.

### 1.1 Alternative: Download Datset form LRZ Sync&Share

We provide a prepared dataset including the rendered views of the Headcam (named: agentview_image) and the rendered eye in hand camera (robot0_eye_in_hand_image) available in LRZ Sync&Share. You need a passwort to enter the folder, which you can get from:
- Maximilian Schindler: maximilian.schindler@tum.de
- Paul Deufel: p.deufel@tum.de
- Niclas Königer: niclas.koeniger@tum.de

You can download it using the following link:
```bash
https://syncandshare.lrz.de/getlink/fiE3JeRoeuWXjov52FKwgW/
```

## 2 Create own dataset

If you want to augment or change the downloaded datasets, follow this guide.

For this project, we take a dataset provided by mimicgen as a base. In this example we will adjust the possible spawn positions of the nut from NutAssemblySqaure (Sqaure_D0) so that it can now spawn close to everywhere on the table, making the policy learn to pick up the nut from a broader variety of spawn postions (In comparison, inside standard Sqaure_D0 the nut ownly spawns in a very limited space on the table). This is especially use full for real world deployment or other experiments.

The dataset resulting from this step can be found in the LRZ share linked in the [README](README.md)

Original spawn area of Square_D0 vs spawn area after expansion:
</br>
<table style="border: none; border-collapse: collapse;">
  <tr style="border: none;">
    <td style="border: none; padding: 10px;">
      <img src="pictures/square_d0_nut_pose.png" width="400" height="267">
    </td>
    <td style="border: none; padding: 10px;">
      <img src="pictures/augmented_5k_nut_pose.png" width="400" height="267">
    </td>
  </tr>
</table>


### 2.1 Preperation
First download a vanilla dataset as descriped in section 1.

Now, you have to prepare this original dataset for the augmentation.

Raise the open file descriptor limit to avoid "Too many open files" or "Could not find resource" errors during training:

In [ ]:
!ulimit -n 65536

In [ ]:
!python mimicgen/mimicgen/scripts/prepare_src_dataset.py \
    --dataset dataset/square_d0.hdf5 \
    --env_interface MG_Square \
    --env_interface_type robosuite

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Got error: No module named 'robosuite_task_zoo'

============= Initialized Observation Utils with Obs Spec =============

using obs modality: low_dim with keys: []
using obs modality: rgb with keys: []

==== Using environment with the following metadata ====
{
    "env_name": "Square_D0",
    "env_version": "1.4.1",
    "type": 1,
    "env_kwargs": {
        "has_renderer": false,
        "has_offscreen_renderer": false,
        "ignore_done": true,
        "use_object_obs": true,
        "use_c

### 2.2 Create new dataset

To create the new dataset that includes a far broader variety of nut spawing positions we defined a new dataset class `Square_RealWorldDeployment(Square_D0)` in `mimicgen/mimicgen/envs/robosuite/nut_assembly.py`.</br>
Its parameters let the nut spawn on nearly the whole table from left to right, but still in the front, close to the robot. Also the bolt moves a little bit to account for real world deployment missplacements.</br></br>
Make sure to use `Square_RealWorldDeployment` as *task name* in the config in the following step!

`generate_dataset.py` generates successfull demonstrations using the changed parameters / new dataset class.</br>
Carefully read the config.json for this.</br>
Generating successfull samples usually takes a few tries. Therefore make sure to set `max_num_failures` and `num_trials` adequately. We got the best results using `"selection_strategy": "nearest_neighbor_object"` together with `"select_src_per_subtask": false`
</br>

In [ ]:
!python mimicgen/mimicgen/scripts/generate_dataset.py \
    --config configs/dataset/square_realworlddeployment.json \
    --source dataset/square_d0.hdf5 \
    --folder dataset/ \
    --num_demos 1

# 3 Create Test and Train set

To train the diffusion policy in the last step you have to define a train / val split. Here 10% validation size were chosen.
YOu have to do this for for every dataset, even the original ones!

In [8]:
!python robomimic/robomimic/scripts/split_train_val.py \
    --dataset dataset/square_d0.hdf5 \
    --ratio 0.1

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
100 validation demonstrations out of 1000 total demonstrations.
Total number of train samples: 138267
Average number of train samples 153.63
Total number of valid samples: 15210
Average number of valid samples 152.1


You can now continue with [DiffusionPolicy Training](DP_Training.ipynb).


---
# Helpfull scripts

## Check datasets or trained policies
<details>
<summary>Overview of dataset</summary>
<br />
Use the following script to get an overview over the dataset:

```bash
python scripts/show_hdf5.py /datasets/DATASETNAME.hdf5
```

As a alternative there is the offical robomimic script, there you can also investigate the *filter keys*: Like split, train, vail, etc....
```bash
python robomimic/robomimic/scripts/get_dataset_info.py --dataset datasets/DATASET_NAME.hdf5
```
<br />
</details>
<br />
<details>
<summary>Playback samples of dataset using simulation</summary>
<br />
Run Robosim to visualize a few samples from the dataset. Use the different flags based on your needs.

```bash
python robomimic/robomimic/scripts/playback_dataset.py \
    --dataset mimicgen/datasets/core/DATASET_NAME.hdf5 \
    --video_path OUTPUT_NAME.mp4 \
    --n 10 \
    --render_image_names agentview robot0_eye_in_hand
```
Remove 'robot0_eye_in_hand' if dataset only contains external camera (and not wrist cam).</br>
Note: `--render_image_names` takes the raw camera names (`agentview`, `robot0_eye_in_hand`, ...) here since the simulator renders live. Only pass the `_image`-suffixed observation keys (e.g. `agentview_image`) together with `--use-obs`, which reads saved pixels straight from the HDF5 instead of re-simulating.
<br />
"Got error: No module named 'robosuite_task_zoo'" can be safely ignored here!
<br />
| Option | Primary Flag(s) | Source of Visuals | Best Used For |
| :--- | :--- | :--- | :--- |
| **1. State Playback** | Default (No special flag) | Rendered live by simulator | Standard simulation dataset sanity checks; inspecting alternative camera views. |
| **2. Image Obs Playback** | `--use-obs` | Saved pixels inside HDF5 | Acts as a pure data reader. Uses observations diretcly from dataset, mo physics step is ever called. |
| **3. Action Playback** | `--use-actions` | Sim execution of saved actions | Verifying simulator physics consistency and ensuring actions match expected behavior. |
| **4. Initial Frames Only** | `--first` | Sim or Obs (First frame only) | Rapidly checking initial object layouts and scene randomization across trajectories. |

</details>
<br />

<details>
<summary>Using test_panda</summary>
<br />
Play back recorded expert demonstrations from the MimicGen HDF5 datasets using the companion script `scripts/test_panda.py`.

Quick examples (run from repository root):

List available demos:
```bash
python scripts/test_panda.py --dataset datasets/square_d0.hdf5 --list
```

Play episode 0 with on-screen rendering (open-loop actions):
```bash
python scripts/test_panda.py --dataset datasets/square_d0.hdf5 --episode 0 --use-actions --render
```

Play episode N headless (no on-screen rendering):
```bash
python scripts/test_panda.py --dataset datasets/square_d0.hdf5 --episode N --use-actions
```

<details>
<summary>Flags:</summary>

- `--episode N` : select demo index (0-based).  
- `--use-actions` : replay stored actions open-loop.  
- `--render` : enable on-screen visualization; omit for headless runs or video capture.
</details>

Notes: ensure the dataset path is correct and required packages (`robomimic`, `robosuite`) are installed. Open an issue if you need help.

</details>
<br />

<details>
<summary>Evaluate trained model</summary>
<br />

```bash
python robomimic/robomimic/scripts/run_trained_agent.py \
    --agent PATH_TO_MODEL.pth \
    --dataset_path /path/to/square_dataset.hdf5 \
    --horizon 400 \
    --n_rollouts 10 \
    --video_path ./eval_model.mp4
```
Only use '--dataset_path' flag if you want to save the rollouts.

Use 'tensorboard --logdir=logs' to run a tensorboard.
</details>

---